In [1]:
"""
report_zero_qty_diagnostic.py
==============================
Answers the question: for commodities with a high rate of Qty=0 rows,
is this a reporting gap (concentrated in specific files, countries, or years)
or random data entry errors scattered across the dataset?

For every affected commodity it produces one Excel workbook with 4 sheets:

  00_Summary        — all affected commodities ranked by severity,
                      with a PATTERN column explaining the likely cause
  01_By_Country     — zero-qty rate per country, for each commodity
  02_By_Period      — zero-qty rate per year-month, for each commodity
  03_Raw_Rows       — every individual zero-qty row for manual review

CONFIG (edit the section below):
  MIN_ZERO_ROWS     — minimum zero-qty rows for a commodity to appear (default 3)
  MIN_ZERO_PCT      — minimum % zero rows to appear (default 5.0)
  EXPORTS_DIR       — path to your raw CSV files

Usage:
  python3 report_zero_qty_diagnostic.py
  Output: Reports/zero_qty_diagnostic.xlsx
"""

import glob
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Environment detection (Colab vs Local) ────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import os as _os
    _os.listdir('/content/drive/MyDrive')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Exports')
    BASE_DIR = Path('/content/drive/MyDrive/Dashboard-BR-CA-Data')
    IN_COLAB = True
except Exception:
    BASE_DIR = Path.cwd().parent / 'Dataset'
    IN_COLAB = False

EXPORTS_DIR  = BASE_DIR / 'Exports'
NORM_FILE    = BASE_DIR / 'Normalised' / 'prices_normalised.csv'
REPORTS_DIR  = BASE_DIR / 'Reports'
LOOKER_DIR   = BASE_DIR / 'Looker'

print(f'Environment : {"Colab" if IN_COLAB else "Local"}')
print(f'Base dir    : {BASE_DIR}')
print(f'Exists      : {BASE_DIR.exists()}')


# ── CONFIG ────────────────────────────────────────────────────
OUT_FILE    = REPORTS_DIR / "zero_qty_diagnostic.xlsx"

MIN_ZERO_ROWS = 3     # ignore commodities with fewer than this many zero-qty rows
MIN_ZERO_PCT  = 5.0   # ignore commodities where < this % of rows are zero-qty

COUNT_UNITS = {"number", "number of dozens", "number of gross",
               "number of pairs", "number of sets", "number of items"}

# ── STYLE HELPERS ─────────────────────────────────────────────
HDR="1F3864"; WHT="FFFFFF"; ALT="EEF4FB"; NTE="DEEAF1"
OK_B="C6EFCE"; OK_F="276221"
WN_B="FFEB9C"; WN_F="9C5700"
FL_B="FFC7CE"; FL_F="9C0006"
CR_B="E26B0A"; CR_F="FFFFFF"   # deep orange — concentrated pattern

def _fill(c): return PatternFill("solid", start_color=c, end_color=c)
def _font(bold=False, color="000000", sz=10):
    return Font(name="Arial", bold=bold, color=color, size=sz)
def _border():
    s = Side(style="thin", color="BDD7EE")
    return Border(left=s, right=s, top=s, bottom=s)
def _left():  return Alignment(horizontal="left",   vertical="center", wrap_text=True)
def _center():return Alignment(horizontal="center", vertical="center", wrap_text=True)
def _right(): return Alignment(horizontal="right",  vertical="center")

def _title(ws, text, ncols):
    ws.append([text] + [""] * (ncols - 1))
    r = ws.max_row
    ws.cell(r, 1).font = _font(bold=True, color=WHT, sz=12)
    ws.cell(r, 1).fill = _fill(HDR)
    ws.cell(r, 1).alignment = _left()
    ws.row_dimensions[r].height = 28
    if ncols > 1:
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=ncols)

def _note(ws, text, ncols):
    ws.append([text] + [""] * (ncols - 1))
    r = ws.max_row
    ws.cell(r, 1).font = _font(color="1F3864")
    ws.cell(r, 1).fill = _fill(NTE)
    ws.cell(r, 1).alignment = _left()
    if ncols > 1:
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=ncols)

def _hdr_row(ws, cols, widths=None):
    ws.append(cols)
    r = ws.max_row
    for c in range(1, len(cols) + 1):
        cell = ws.cell(r, c)
        cell.font = _font(bold=True, color=WHT)
        cell.fill = _fill(HDR)
        cell.alignment = _center()
        cell.border = _border()
    if widths:
        for c, w in enumerate(widths, 1):
            ws.column_dimensions[get_column_letter(c)].width = w
    ws.row_dimensions[r].height = 30
    return r

def _data_row(ws, vals, alt=False, color_col=None, bg_override=None, fg_override=None):
    ws.append(vals)
    r = ws.max_row
    bg = bg_override if bg_override else (ALT if alt else WHT)
    for c, v in enumerate(vals, 1):
        cell = ws.cell(r, c)
        cell.border = _border()
        cell.alignment = _left()
        if c == color_col and bg_override:
            cell.fill = _fill(bg_override)
            cell.font = _font(bold=True, color=fg_override or "000000")
        else:
            cell.fill = _fill(ALT if alt else WHT)
            cell.font = _font()

def _pct_cell(ws, row, col, value):
    """Write a percentage cell with colour scale: green < 10, amber 10-30, red > 30."""
    cell = ws.cell(row, col, value=value)
    cell.border = _border()
    cell.alignment = _right()
    if value == 0:
        cell.fill = _fill(OK_B); cell.font = _font(bold=False, color=OK_F)
    elif value < 10:
        cell.fill = _fill(WHT); cell.font = _font()
    elif value < 30:
        cell.fill = _fill(WN_B); cell.font = _font(bold=True, color=WN_F)
    else:
        cell.fill = _fill(FL_B); cell.font = _font(bold=True, color=FL_F)

def _autofilter(ws, header_row, ncols):
    ws.auto_filter.ref = f"A{header_row}:{get_column_letter(ncols)}{header_row}"

# ── LOAD ──────────────────────────────────────────────────────
def load_all(files):
    frames, errors = [], []
    for f in files:
            df = pd.read_csv(f, skiprows=1, dtype=str)
            df.columns = df.columns.str.strip()
            df = df[pd.to_datetime(df.get("Period", pd.Series()), errors="coerce").notna()].copy()
            df["_source_file"] = Path(f).name
            frames.append(df)
        except Exception as e:
            errors.append(Path(f).name)
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return combined, errors

# ── ANALYSIS ──────────────────────────────────────────────────
def prepare(raw):
    df = raw.copy()
    df["_q"]     = pd.to_numeric(df.get("Quantity"),   errors="coerce")
    df["_v"]     = pd.to_numeric(df.get("Value ($)"),  errors="coerce")
    df["_unit"]  = df.get("Unit of measure", pd.Series("")).astype(str).str.strip().str.lower()
    df["_blank"] = df["_unit"] == "blank"
    df["_period"]= pd.to_datetime(df.get("Period"), errors="coerce")
    df["_year"]  = df["_period"].dt.year
    df["_ym"]    = df["_period"].dt.to_period("M").astype(str)
    df["_zero"]  = (df["_q"] == 0) & ~df["_blank"] & (df["_v"] > 0)
    return df


def commodity_summary(df):
    """One row per commodity that meets the thresholds."""
    if "Commodity" not in df.columns:
        return pd.DataFrame()

    rows = []
    for comm, grp in df.groupby("Commodity"):
        total   = len(grp)
        zero    = int(grp["_zero"].sum())
        if zero < MIN_ZERO_ROWS:
            continue
        pct = round(zero / total * 100, 1)
        if pct < MIN_ZERO_PCT:
            continue

        dom_unit   = grp["_unit"].value_counts().index[0] if len(grp) else "—"
        is_count   = dom_unit in COUNT_UNITS
        total_val  = int(grp.loc[grp["_zero"], "_v"].sum())
        countries  = grp.loc[grp["_zero"], "Country"].nunique() if "Country" in grp.columns else 0
        n_files    = grp.loc[grp["_zero"], "_source_file"].nunique()

        # Determine concentration pattern
        # Country concentration: top country > 70% of zero rows
        if "Country" in grp.columns:
            top_country_pct = (grp[grp["_zero"]].groupby("Country").size()
                               .sort_values(ascending=False).iloc[0] / zero * 100
                               if zero else 0)
        else:
            top_country_pct = 0

        # File concentration: top file > 70% of zero rows
        top_file_pct = (grp[grp["_zero"]].groupby("_source_file").size()
                        .sort_values(ascending=False).iloc[0] / zero * 100
                        if zero else 0)

        # Year concentration: single year > 80% of zero rows
        top_year_pct = (grp[grp["_zero"]].groupby("_year").size()
                        .sort_values(ascending=False).iloc[0] / zero * 100
                        if zero else 0)

        if top_file_pct >= 70:
            pattern = "📁 Concentrated in 1 file — likely a reporting gap"
        elif top_country_pct >= 70:
            pattern = "🌍 Concentrated in 1 country — may be country-specific reporting"
        elif top_year_pct >= 80:
            pattern = "📅 Concentrated in 1 year — possible policy/format change"
        elif n_files == 1:
            pattern = "📁 Only 1 file affected"
        elif pct >= 50 and is_count:
            pattern = "⚠️ High rate + count unit — likely systematic data entry error"
        elif pct >= 50:
            pattern = "⚠️ High rate — investigate further"
        else:
            pattern = "🔍 Scattered — random entry errors likely"

        rows.append({
            "Commodity":         comm,
            "Unit":              dom_unit.title(),
            "Unit Type":         "⚠️ Count" if is_count else "Weight/Volume/Other",
            "Total Rows":        total,
            "Zero Qty Rows":     zero,
            "% Zero":            pct,
            "Countries Affected":countries,
            "Files Affected":    n_files,
            "Value $ Excluded":  total_val,
            "Pattern":           pattern,
        })

    return (pd.DataFrame(rows)
              .sort_values(["Zero Qty Rows", "Value $ Excluded"], ascending=False)
              .reset_index(drop=True))


def by_country(df, comm_list):
    """Per-country breakdown for each affected commodity."""
    if "Country" not in df.columns:
        return pd.DataFrame()
    rows = []
    for comm in comm_list:
        grp = df[df["Commodity"] == comm]
        for country, sub in grp.groupby("Country"):
            total = len(sub)
            zero  = int(sub["_zero"].sum())
            val   = int(sub.loc[sub["_zero"], "_v"].sum()) if zero else 0
            rows.append({
                "Commodity": comm,
                "Country":   country,
                "Total Rows": total,
                "Zero Qty Rows": zero,
                "% Zero": round(zero / total * 100, 1) if total else 0,
                "Value $ Excluded": val,
            })
    return (pd.DataFrame(rows)
              .sort_values(["Commodity", "Zero Qty Rows"], ascending=[True, False])
              .reset_index(drop=True))


def by_period(df, comm_list):
    """Per year-month breakdown for each affected commodity."""
    rows = []
    for comm in comm_list:
        grp = df[df["Commodity"] == comm]
        for ym, sub in grp.groupby("_ym"):
            total = len(sub)
            zero  = int(sub["_zero"].sum())
            val   = int(sub.loc[sub["_zero"], "_v"].sum()) if zero else 0
            rows.append({
                "Commodity":   comm,
                "Period":      ym,
                "Total Rows":  total,
                "Zero Qty Rows": zero,
                "% Zero": round(zero / total * 100, 1) if total else 0,
                "Value $ Excluded": val,
            })
    return (pd.DataFrame(rows)
              .sort_values(["Commodity", "Period"])
              .reset_index(drop=True))


def raw_rows(df, comm_list):
    """All individual zero-qty rows for affected commodities."""
    mask = df["_zero"] & df["Commodity"].isin(comm_list)
    cols = ["Commodity", "Period", "Country", "Province", "State",
            "Quantity", "Unit of measure", "Value ($)", "_source_file"]
    cols = [c for c in cols if c in df.columns]
    return (df[mask][cols]
              .sort_values(["Commodity", cols[1] if len(cols) > 1 else "Commodity"])
              .reset_index(drop=True))


# ── BUILD WORKBOOK ─────────────────────────────────────────────
def build(df, run_dt, total_files):
    summary_df   = commodity_summary(df)
    comm_list    = list(summary_df["Commodity"]) if not summary_df.empty else []
    country_df   = by_country(df, comm_list)
    period_df    = by_period(df, comm_list)
    raw_df       = raw_rows(df, comm_list)

    wb = Workbook()
    wb.remove(wb.active)

    # ── 00 Summary ────────────────────────────────────────────
    ws = wb.create_sheet("00_Summary")
    ws.sheet_view.showGridLines = False
    ncols = 10
    _title(ws, "ZERO QTY DIAGNOSTIC — WHERE ARE MISSING QUANTITIES CONCENTRATED?", ncols)
    ws.append([""])
    _note(ws, f"Generated: {run_dt}   |   Files loaded: {total_files:,}   |   "
              f"Affected commodities: {len(summary_df)}   |   "
              f"Thresholds: ≥ {MIN_ZERO_ROWS} zero rows AND ≥ {MIN_ZERO_PCT}% of commodity rows", ncols)
    ws.append([""])
    _note(ws, "PATTERN KEY:  📁 Concentrated in 1 file = reporting gap, investigate that file  |  "
              "🌍 1 country = country-specific issue  |  📅 1 year = format change  |  "
              "⚠️ High rate = systematic  |  🔍 Scattered = random errors", ncols)
    ws.append([""])

    if summary_df.empty:
        _note(ws, f"No commodities found meeting thresholds (≥{MIN_ZERO_ROWS} rows, ≥{MIN_ZERO_PCT}%).", ncols)
    else:
        cols0 = list(summary_df.columns)
        widths0 = [58, 26, 16, 12, 14, 10, 20, 16, 26, 44]
        hr = _hdr_row(ws, cols0, widths0[:len(cols0)])
        _autofilter(ws, hr, len(cols0))

        for i, (_, r) in enumerate(summary_df.iterrows()):
            vals = list(r)
            ws.append(vals)
            rn = ws.max_row
            alt = i % 2 == 1
            for c, v in enumerate(vals, 1):
                cell = ws.cell(rn, c)
                cell.border = _border()
                cell.alignment = _left()
                # % Zero column (col 6) — colour scale
                if c == 6 and isinstance(v, (int, float)):
                    _pct_cell(ws, rn, c, v)
                # Unit Type column (col 3) — amber for count
                elif c == 3 and isinstance(v, str) and "Count" in v:
                    cell.fill = _fill(WN_B); cell.font = _font(bold=True, color=WN_F)
                # Pattern column (col 10) — orange for concentrated
                elif c == 10 and isinstance(v, str) and ("Concentrated" in v or "High rate" in v):
                    cell.fill = _fill(CR_B); cell.font = _font(bold=True, color=CR_F)
                else:
                    cell.fill = _fill(ALT if alt else WHT); cell.font = _font()

        ws.freeze_panes = f"A{hr + 1}"

    # ── 01 By Country ─────────────────────────────────────────
    ws1 = wb.create_sheet("01_By_Country")
    ws1.sheet_view.showGridLines = False
    _title(ws1, "ZERO QTY ROWS BY COUNTRY", 6)
    ws1.append([""])
    _note(ws1, "A country with a very high % Zero compared to others suggests a country-specific reporting issue. "
               "If one country accounts for most zero-qty rows for a commodity, the problem is likely at that trade partner's reporting level.", 6)
    ws1.append([""])

    if country_df.empty:
        _note(ws1, "No data.", 6)
    else:
        cols1 = list(country_df.columns)
        hr1 = _hdr_row(ws1, cols1, [58, 22, 12, 14, 10, 22])
        _autofilter(ws1, hr1, len(cols1))
        prev_comm = None
        alt = False
        for _, r in country_df.iterrows():
            if r["Commodity"] != prev_comm:
                prev_comm = r["Commodity"]
                alt = False
            vals = list(r)
            ws1.append(vals)
            rn = ws1.max_row
            for c, v in enumerate(vals, 1):
                cell = ws1.cell(rn, c)
                cell.border = _border()
                cell.alignment = _left()
                if c == 5 and isinstance(v, (int, float)):
                    _pct_cell(ws1, rn, c, v)
                else:
                    cell.fill = _fill(ALT if alt else WHT); cell.font = _font()
            alt = not alt
        ws1.freeze_panes = f"A{hr1 + 1}"

    # ── 02 By Period ──────────────────────────────────────────
    ws2 = wb.create_sheet("02_By_Period")
    ws2.sheet_view.showGridLines = False
    _title(ws2, "ZERO QTY ROWS BY YEAR-MONTH", 6)
    ws2.append([""])
    _note(ws2, "A sudden spike in % Zero at a particular period suggests a reporting format change or data extraction issue at that time. "
               "Consistent high % across all periods suggests the commodity is systematically under-reported.", 6)
    ws2.append([""])

    if period_df.empty:
        _note(ws2, "No data.", 6)
    else:
        cols2 = list(period_df.columns)
        hr2 = _hdr_row(ws2, cols2, [58, 14, 12, 14, 10, 22])
        _autofilter(ws2, hr2, len(cols2))
        prev_comm = None
        alt = False
        for _, r in period_df.iterrows():
            if r["Commodity"] != prev_comm:
                prev_comm = r["Commodity"]
                alt = False
            vals = list(r)
            ws2.append(vals)
            rn = ws2.max_row
            for c, v in enumerate(vals, 1):
                cell = ws2.cell(rn, c)
                cell.border = _border()
                cell.alignment = _left()
                if c == 5 and isinstance(v, (int, float)):
                    _pct_cell(ws2, rn, c, v)
                else:
                    cell.fill = _fill(ALT if alt else WHT); cell.font = _font()
            alt = not alt
        ws2.freeze_panes = f"A{hr2 + 1}"

    # ── 03 Raw Rows ───────────────────────────────────────────
    ws3 = wb.create_sheet("03_Raw_Rows")
    ws3.sheet_view.showGridLines = False
    _title(ws3, "ALL ZERO QTY ROWS — FOR MANUAL REVIEW", len(raw_df.columns) if not raw_df.empty else 6)
    ws3.append([""])
    _note(ws3, "Every individual row where Qty=0, unit is not Blank, and Value > 0. "
               "Sort by Commodity + Value to find the highest-value missing quantities.", len(raw_df.columns) if not raw_df.empty else 6)
    ws3.append([""])

    if raw_df.empty:
        _note(ws3, "No rows found.", 6)
    else:
        ncols3 = len(raw_df.columns)
        col_widths = {"Commodity":58,"Period":14,"Country":22,"Province":14,
                      "State":14,"Quantity":12,"Unit of measure":26,"Value ($)":14,"_source_file":30}
        widths3 = [col_widths.get(c, 16) for c in raw_df.columns]
        hr3 = _hdr_row(ws3, list(raw_df.columns), widths3)
        _autofilter(ws3, hr3, ncols3)
        for i, (_, r) in enumerate(raw_df.iterrows()):
            _data_row(ws3, list(r), alt=i % 2 == 1)
        ws3.freeze_panes = f"A{hr3 + 1}"

    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    wb.save(OUT_FILE)
    return summary_df


# ── MAIN ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "=" * 58)
    print("  ZERO QTY DIAGNOSTIC")
    print("=" * 58)
    print(f"\n  Thresholds: ≥ {MIN_ZERO_ROWS} zero rows AND ≥ {MIN_ZERO_PCT}% of commodity rows")

    files = glob.glob(str(EXPORTS_DIR / "**/*.csv"), recursive=True)
    print(f"\n  Found {len(files)} CSV files in {EXPORTS_DIR}")
    if not files:
        print("  ERROR: No files found. Check EXPORTS_DIR path.")
        exit(1)

    print("  Loading all files...")
    raw, load_errors = load_all(files)
    n_files = raw["_source_file"].nunique() if not raw.empty else 0
    print(f"  Loaded {len(raw):,} rows from {n_files} files")
    if load_errors:
        print(f"  {len(load_errors)} files failed: {', '.join(load_errors[:5])}")

    print("  Analysing zero-qty patterns...")
    df = prepare(raw)
    run_dt = datetime.now().strftime("%Y-%m-%d %H:%M")
    summary = build(df, run_dt, n_files)

    if summary.empty:
        print(f"\n  No commodities found meeting thresholds.")
    else:
        print(f"\n  Found {len(summary)} affected commodities")
        print(f"\n  Top 5 by zero-qty rows:")
        for _, r in summary.head(5).iterrows():
            print(f"    {r['Commodity'][:60]:<62} "
                  f"{r['Zero Qty Rows']:>4} rows ({r['% Zero']:>5.1f}%)  "
                  f"${r['Value $ Excluded']:>8,}  {r['Pattern']}")

    print(f"\n  Saved: {OUT_FILE}\n")


  ZERO QTY DIAGNOSTIC

  Thresholds: ≥ 3 zero rows AND ≥ 5.0% of commodity rows

  Found 158 CSV files in /Users/midorikawaguti/DevProject/Dashboard-BR-CA/Dataset/Exports
  Loading all files...
  Loaded 2,562,365 rows from 158 files
  Analysing zero-qty patterns...

  Found 175 affected commodities

  Top 5 by zero-qty rows:
    4202.21.00 - Handbags, with outer surface of leather or of c    420 rows ( 45.3%)  $ 936,041  🌍 Concentrated in 1 country — may be country-specific reporting
    2933.59.00 - Heterocyclic compd,w nitr hetero-atoms,cont pyr    391 rows ( 48.5%)  $ 358,798  🔍 Scattered — random entry errors likely
    4811.41.00 - Paper & paperboard, self-adhesive, in rolls or     321 rows ( 26.6%)  $1,383,634  🔍 Scattered — random entry errors likely
    1001.99.90 - Wheat, nes and meslin, o/t certified organic, o    301 rows (  9.3%)  $  59,179  🌍 Concentrated in 1 country — may be country-specific reporting
    2933.99.00 - Heterocyclic compounds with nitrogen hetero-ato    2